In [23]:
# imports

# show rich formats 
from IPython.display import display, Markdown

# get python to interact with openai
from openai import OpenAI

# use personal openai key
import os
from dotenv import load_dotenv

# check load_dotenv works - should come back 'True'
# load_dotenv()

# make a destination 'resumes' directory for the work

os.makedirs("resumes", exist_ok=True)

# use python to format markdown to html
from markdown import markdown

#### because we're sending this out as a resume, we want it to be in a .pdf format. 
#### that means we'll have to use a library called weasy... 
##### <i><b>< record scratch ></i></b>
#### nope. sorry. 
#### not using weasyprint. lining weasyprint libraries up with each particular python environmet and setting / dedicating kernals still causes nightmares of 'public-speaking-in-underpants' proportions. surely a powerful tool, but it's got the playfulness and ease of use of a rabid porcupine. <i>no grazie</n>.
#### instead, going with pdfkit. working in html, so it'll do the job.
##### * note: pdfkit works using wkhtmltopdf, which <i>in very simple terms</i> converts a webpage to a pdf file. please see [reference.txt](https://github.com/npj210mlk/jobapp_prompter/blob/main/requirements.txt) in the repo for installation instructions.


In [2]:
# import pdfkit

# # test
# pdfkit.from_string("<h1>It should be called 'QueezyPrint,' amirite?</h1>", "output.pdf")

ModuleNotFoundError: No module named 'pdfkit'

#### ha! apparently, jupyter agrees - came back 'True'

***
#### with the libraries imported we can now break the project down into four (4) steps:
##### 1.) open and read the markdown resume file
#####     * see requirements notes
##### 2.) input the desired job description
##### 3.) template some prompt engineering
##### 4.) convert markdown to html
##### 5.) convert html to pdf
***
*** 
#### Step 1: Open and Read the Markdown Resume
****

In [20]:
# open resume and read it
with open("/Users/nicholasjoseph/Desktop/nj_res.md", "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

# check to see if it worked:
# display(Markdown(resume_string))

# markdown resume displays as expected.

***
#### Step 2: Input the Desired Job Description
***

In [3]:
# job description will be from anywhere, so input is used to pause the program until we find one to copy/paste into this variable 
job_desc_string = input()

 About the job Enlighten, honored as a Top Workplace from USA Today, is a leader in big data solution development and deployment, with expertise in cloud-based services, software and systems engineering, cyber capabilities, and data science. Enlighten provides continued innovation and proactivity in meeting our customers’ greatest challenges.  We recognize that the most effective environment for your projects doesn’t always look the same. Our hybrid work approach ensures that you can make lasting relationships with your team and collaborate in-person to get the job done—while having the flexibility to be working from home when needed to achieve focused results.  Why Enlighten?  Benefits  At Enlighten, our team’s unwavering work ethic, top talent and celebration of innovative ideas have helped us thrive. We know that our employees are essential to our company’s success, so we seek to take care of you as much as you take care of us. Here are a few highlights of our benefits package:   10

***
#### Step 3: Template Some Prompt Engineering
##### - this involves dealing with ChatGPT, particularly the ChatGPT 4o-mini pre-trained transformer.
##### <u>a couple of things about the ChatGPT 4o-mini engine (model)</u>:
#####     a.) it is the smaller, more affordable version of the massive GPT-4o engine available to developers; and
#####     b.) because its focus is on text classificaton / prediction, it is purely a "decoder-only" type transformer
#### The idea is to have ChatGPT create the prompt to respond to the likely Applicant Tracking System being used by the job poster.
##### Reason being, machines talk to machines better than we can. 
***

In [7]:
# making up a random "lambda" function to create the variable "prompt_template"

# the text is what I would be putting into ChatGPT were I to do this one job at a time - prompt engineering

prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

4. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.

5.) **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

In [8]:
# set the prompt variable for our ChatGPT message roles
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 4: Generate the Resume with GPT-4o-mini
##### - Make the api call and tell GPT to write the resume using the prompt from above
***

In [9]:
# set up api client
open_apikey = os.environ.get("openapi_apikey")

client = OpenAI(api_key = open_apikey)

# make the call

# set response variable to hold the all info we get back
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    # set our roles up - think of casting a play: "Today, the role of the Expert Resume Writer will be played by the system."
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
    
    # give it some creative license 
    temperature = 0.7
)

# get our response
response_string = response.choices[0].message.content

***
#### Step 5: Show Us the Results
***

In [10]:
# separate new resume from improvements AI suggests at 'Additional Suggestions'
response_list = response_string.split("## Additional Suggestions")

In [11]:
display(Markdown(response_list[0]))

# Nicholas Joseph  
**Product Manager**  
📍 San Antonio, TX | Remote | Hybrid | On-Site within 20-mile radius of SATX  
📧 [nickpjoseph210@gmail.com](mailto:nickpjoseph210@gmail.com)  
📞 (210) 771-9853  
🔗 [LinkedIn](http://linkedin.com/in/nickjosephsanantonio)

---

## 🧠 Career Summary

- **Data Engineering & Analytics:** 3 years of experience in developing scalable data solutions, enhancing reporting accuracy, and enabling data-driven decision-making.
- **Project Management:** 7 years of experience leading cross-functional teams in Agile and Waterfall methodologies, specializing in technical project delivery and stakeholder engagement.
- **Business Development:** 15 years of experience fostering relationships and driving growth through data-driven strategies in customer-facing environments.

---

## 🌟 Impact Summary

- **Cloud Data Engineer, Spaulding Ridge:** Developed data pipelines for major sports leagues, improving data extraction efficiency by 98.9% and reporting accuracy by 54%.
- **Project Manager, NobleHands:** Led 16+ projects with zero budget overages, achieving consistent Five-Star ratings on customer platforms.
- **Junior Data Engineer, Apexon:** Successfully migrated large-scale banking data to cloud environments while serving as a subject matter expert and trainer, enhancing team capabilities.

---

## 💼 Professional Experience

### Cloud Data Engineer — *Spaulding Ridge (formerly Data Clymer)*  
**Remote | 07/2022–06/2023**  
- Designed and implemented data pipelines for the NFL's *Fan 360 Experience*, enabling real-time analytics for enhanced user engagement.
- Collaborated with Sales and Marketing to align data solutions with business strategies, ensuring the seamless integration of analytics into operational workflows.
- **Skills:** API Management, ETL, Agile, Data Analytics, Stakeholder Collaboration

---

### Project Manager — *NobleHands H & C*  
**San Antonio, TX | 10/2017–01/2020, 07/2023–01/2025**  
- Oversaw project delivery for multiple initiatives, achieving 100% safety compliance and maintaining high customer satisfaction levels.
- Cultivated relationships with stakeholders to drive project alignment with business objectives and enhance service delivery.
- **Skills:** Agile Project Management, Stakeholder Engagement, Conflict Resolution

---

### Junior Data Engineer — *Apexon (formerly Gathi Analytics)*  
**Remote | 12/2020–03/2022**  
- Led the migration of extensive banking datasets to cloud infrastructure, supporting operational efficiency and data accessibility.
- Delivered training sessions to new hires, improving team performance and knowledge transfer.
- **Skills:** Data Migration, Cross-Functional Collaboration, Agile (Scrum/Kanban)

---

## 📜 Certifications & Education

- **Professional Scrum Manager I** — *scrum.org* — *Exp. 12/2024*  
- **Project Management Professional (in progress)** — *Technical Institute of America, 2024*  
- **Data Science Certificate** — *Codeup, San Antonio TX | 2020*  
- **B.S. in Biology** — *Concordia University, Austin TX*

---

## 🔗 Relevant Links

- [Spaulding Ridge Acquisition](https://www.spauldingridge.com/articles/spaulding-ridge-acquires-data-clymer-to-expand-data-solutions-offerings-and-strengthen-tech-partnerships/)
- [Apexon](https://www.apexon.com/)
- [Scrum.org](http://scrum.org)

---



***
#### Step 6: Save the New Resume
##### Great. Hit several snags. You either need WeasyPrint installed in some capacity, or other tools I found (like 'Grip') are difficult to automate and test on MacOS. 
##### Looks like the programming gods have spoken: we're going with WeasyPrint.
##### <center><span style ="color:red"><b><u>To Do This:</u></b></span><center>
##### 1.) completely uninstall existing WeasyPrint's existence from your machine with pip and brew uninstalls;
##### 2.) run a 'brew cleanup' just to wipe out any remnants;
##### 3.) form home directory in Terminal (I'm using Mac), type 'brew install cairo pango gdk-pixbuf libffi' - these are the native Weasyprint dependencies;
##### 4.) set your environment variables in your profile (for me, my ~/.zshrc file) to the following:
###### export DYLD_LIBRARY_PATH=/opt/homebrew/lib:$DYLD_LIBRARY_PATH
###### export LIBRARY_PATH="/opt/homebrew/lib:$LIBRARY_PATH"
###### export PKG_CONFIG_PATH="/opt/homebrew/lib/pkgconfig:/opt/homebrew/share/pkgconfig"
###### export PATH="/opt/homebrew/bin:$PATH"
##### 5.) restart the terminal;
##### 6.) navigate to your project folder;
##### 7.) type 'pip install weasyprint markdown; and (finally)
##### 8.) open your notebook from your project file with 'jupyter notebook'
***

In [25]:
# Markdown was already imported earlier
# import WeasyPrint and its HTML abilities
from weasyprint import HTML

# save it as PDF
output_pdf_file = "resumes/nick_joseph_resume.pdf"

# convert the Markdown file to HTML
html_content = markdown(response_list[0])

# now take that HTML and convert it into a PDF and save
HTML(string=html_content).write_pdf(output_pdf_file)

In [26]:
# let's save the new Markdown file, too
markdown_output = "resumes/resume_new_markdown.md"

with open(markdown_output, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

In [ ]:
# # markdown is already imported
# from weasyprint import HTML

In [ ]:
# from markdown2pdf import convert

# markdown_resume = display(Markdown(response_list[0]))

# convert(markdown_resume, "nj_resume_in_pdf.pdf")